# 05 - Grenoble cycling network (Section VI)

Real Grenoble cycling network (3,539 nodes, 5,065 edges). Comparison of the algorithms (Fig. 9) and computation times (Table VII).

**Runtime warning:** with the paper settings (80 generations x 50 individuals, 20 Bayesian calls) the GA and Bayesian runs take **many hours** (up to ~1.5 h per budget and method). Results are cached in `results/<preset>/`, so this only has to be done once. For a quick check set `QUICK = True` (5 generations, 10 Bayesian calls) - the numbers will then be worse than with the full settings.

**Choose the model variant** (see README, section *Two model variants*):

* `"dos"` (default) - integer DoS weights, linear models in $s_\mathcal{G}$ and $l_\mathcal{G}$ (ITSC-2024 calibration), exponential cost per metre.
* `"cscs"` - CycleRAP/CSCS weights, NNLS model $\tilde\theta=[0.41, 0.58, 0.45, 0.57]$ of Eq. (10), Cerema costs.
  With this model safety and comfort percentages are identical, because the prediction only depends on $l_\mathcal{G}$.

The GA and the Bayesian optimiser are stochastic: their results vary from run to run (seed fixed below for repeatability). MLCC and the brute-force optimum are deterministic.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # the cycling_tits package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from cycling_tits import CITIES, FIGURES, RESULTS, load_network, load_survey, pretty, Model, alphas
FIGURES.mkdir(exist_ok=True)

In [ ]:
from cycling_tits.experiments import compare_algorithms, percents, ga_histories, time_table
from cycling_tits.plotting import comparison_bars, ga_evolution

PRESET = "dos"                # or "cscs"
TARGETS = ["safety", "comfort"]
METHODS = ['MLCC', 'SGA', 'SGA-MLCC', 'Bayesian']
QUICK = False                  # True: very short GA / Bayesian runs, only to test the notebook
GENERATIONS, POP_SIZE, BAYES_CALLS = (5, 10, 10) if QUICK else (80, 50, 20)
SEED = 0
TAG = "_quick" if QUICK else ""
OUT = FIGURES / PRESET; OUT.mkdir(exist_ok=True)

In [ ]:
results = {t: compare_algorithms("Grenoble", PRESET, t, METHODS, generations=GENERATIONS, pop_size=POP_SIZE,
                                bayes_calls=BAYES_CALLS, seed=SEED, tag=TAG)
           for t in TARGETS}

## % of achievable improvement  $100\,|\hat y_{\omega_0}-\hat y(\omega^*)| / |\hat y_{FS}-\hat y_{\omega_0}|$

In [ ]:
tables = {t: pd.DataFrame({m: percents(results[t], m) for m in METHODS}, index=[f"i={i}" for i in range(10)]).T.round(2)
          for t in TARGETS}
tables["safety"]

In [ ]:
tables["comfort"]

## Fig. 9 (a) - comparison of the algorithms (plain = safety, hatched = comfort)

In [ ]:
comparison_bars({m: percents(results["safety"], m) for m in METHODS},
                {m: percents(results["comfort"], m) for m in METHODS},
                outfile=OUT / "maximize_safety_Grenoble_comparasion.png", methods=METHODS);

## Fig. 9 (b), (c) - evolution of SGA and SGA-MLCC over the generations (safety)

In [ ]:
ga_evolution(ga_histories(results["safety"], "SGA"), ga_histories(results["safety"], "SGA-MLCC"),
             outfile=OUT / "Grenoble_comparation_GA_LCC_safety.png",
             legend_outfile=OUT / "Grenoble_comparation_GA_LCC_safety_legend.png");

## Table VII - computation time

In [ ]:
time_table(results["safety"], METHODS)